In [1]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# 1. 입력/출력 문장 설정
encoder_tokens = "What is your name ?".split()
decoder_tokens = "My name is John .".split()

# 2. 실제 번역 흐름에 가까운 어텐션 점수 수동 설정
attention_scores = [
    [0.1,   0.2,  2.2,  1.2,  0.0],  # "My"   → 대명사 대응: "your" ↑, 보조로 "name"
    [0.1,   0.1,  0.3,  3.0,  0.0],  # "name" → 입력문 핵심에 대응: "name" ↑↑
    [0.1,   2.2,  0.2,  0.7,  0.0],  # "is"   → 동사 대응: "is" ↑, 보조로 "name"
    [0.0,   0.1,  0.2,  0.8,  0.0],  # "John" → 원천정보는 인코더에 없음: "name"에 약간
    [0.1,   0.1,  0.1,  0.2,  1.5],  # "."    → 구두점/문장 경계: "?"에 상대적 집중
]

# 3. softmax 함수로 어텐션 가중치 계산
def softmax(x):
    e = np.exp(x - np.max(x))
    return e / np.sum(e)

attention_all = np.array([softmax(score) for score in attention_scores])  # shape: [decoder_step, encoder_step]

# 4. 어텐션 화살표 시각화 함수
def plot_attention_arrows(decoder_step):
    weights = attention_all[decoder_step]

    plt.figure(figsize=(10, 4))
    y_enc = -1
    y_dec = 1

    # 디코더 단어 (위쪽)
    for i, word in enumerate(decoder_tokens):
        if i == decoder_step:
            plt.text(i, y_dec, word, fontsize=14, ha='center', va='bottom', fontweight='bold', color='black')

    # 인코더 단어 (아래쪽)
    for i, word in enumerate(encoder_tokens):
        plt.text(i, y_enc, word, fontsize=12, ha='center', va='top')

    # 어텐션 화살표 그리기
    for i, weight in enumerate(weights):
        plt.annotate("",
                     xy=(i, y_enc + 0.2), xycoords='data',
                     xytext=(decoder_step, y_dec - 0.2), textcoords='data',
                     arrowprops=dict(
                         arrowstyle="->",
                         color='blue',
                         lw=weight * 6,                    # 화살표 굵기
                         alpha=min(weight * 2, 1.0)        # 투명도 제한
                     ))

    plt.title(f'Attention from Decoder Word "{decoder_tokens[decoder_step]}"')
    plt.axis('off')
    plt.xlim(-1, max(len(encoder_tokens), len(decoder_tokens)))
    plt.ylim(y_enc - 1, y_dec + 1)
    plt.tight_layout()
    plt.show()
    plt.close()  # 중복 출력 방지

# 5. 드롭다운 UI
decoder_dropdown = widgets.Dropdown(
    options=[(f"{i+1}. {word}", i) for i, word in enumerate(decoder_tokens)],
    description='Decoder:',
    layout=widgets.Layout(width='50%')
)
output = widgets.Output()

def on_decoder_change(change):
    output.clear_output(wait=True)
    with output:
        plot_attention_arrows(change['new'])

decoder_dropdown.observe(on_decoder_change, names='value')

# 6. 실행
display(decoder_dropdown, output)
with output:
    clear_output(wait=True)
    plot_attention_arrows(0)


Dropdown(description='Decoder:', layout=Layout(width='50%'), options=(('1. My', 0), ('2. name', 1), ('3. is', …

Output()